In [1]:
print("all ok")

all ok


In [3]:
%pip install -U langgraph

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/572.8 kB ? eta -:--:--
   ---------------------------------------- 572.8/572.8 kB 10.0 MB/s  0:00:00
   ---------------------------------------- 0.0/813.1 kB ? eta -:--:--
   ---------------------------------------- 813.1/813.1 kB 36.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ------------------------------ --------- 1.6/2.0 MB 4.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.0 MB 2.7 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 1.9 MB/s  0:00:00

   - --------------------------------------  1/36 [xxhash]
   -- -------------------------------------  2/36 [websockets]
   -- -------------------------------------  2/36 [websockets]

In [4]:
import uuid
from dataclasses import dataclass
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)
from langgraph.runtime import Runtime

In [5]:
import sys
print(sys.executable)

C:\Users\sharm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [6]:
%pip show langgraph

Name: langgraph
Version: 1.2.14
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: C:\Users\sharm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys
!{sys.executable} -m pip install -U langgraph

Defaulting to user installation because normal site-packages is not writeable


In [10]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0
)

ImportError: Initializing ChatOpenAI requires the langchain-openai package. Please install it with `pip install langchain-openai`

In [9]:
%pip install -U langchain

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install -U langchain-openai

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 24.4 MB/s  0:00:00
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/941.2 kB ? eta -:--:--
   ---------------------------------------- 941.2/941.2 kB 2.2 MB/s  0:00:00

   ---------------------------------------- 0/5 [regex]
   ---------------------------------------- 0/5 [regex]
   ---------------- ----------------------- 2/5 [tiktoken]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 3/5 [openai]
   ------------------------ --------------- 

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [13]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

In [14]:
@dataclass
class Context:
    user_id: str

In [15]:
Context.user_id=str(uuid.uuid4())

In [16]:
Context.user_id

'45027464-c323-429c-b61a-d4533bd00e5b'

In [18]:
def assistant_node(
    state: MessagesState,
    runtime: Runtime[Context]
):
    user_id = runtime.context.user_id
    namespace = ("users", user_id, "memories")
    user_message = state["messages"][-1].content
    
    if user_message.lower().startswith("remember:"):
        memory = user_message[len("remember:"):].strip()
        runtime.store.put(
            namespace,
            str(uuid.uuid4()),
            {
                "data": memory
            }
        )
        print(
            f"[MEMORY SAVED]: {memory}"
        )
        
    memories = runtime.store.search(namespace)
    memory_text = "\n".join(
        memory.value["data"]
        for memory in memories
    )
    if not memory_text:
        memory_text = "No saved memories."
        
    
    system_prompt = f"""
    You are a helpful assistant.

    Long-term memories about this user:

    {memory_text}
    """


    response = model.invoke(
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *state["messages"]
        ]
    )
    return {
        "messages": [response]
    }


In [19]:
builder = StateGraph(
    MessagesState,
    context_schema=Context
)

In [20]:
builder.add_node(
    "assistant",
    assistant_node
)

In [21]:
builder.add_edge(
    START,
    "assistant"
)

builder.add_edge(
    "assistant",
    END
)

In [22]:
graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [23]:
def chat(
    message,
    user_id,
    thread_id
):

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        config={
            "configurable": {
                "thread_id": thread_id
            }
        },

        context=Context(
            user_id=user_id
        )
    )


    answer = result["messages"][-1].content

    print("\nUSER:")
    print(message)

    print("\nASSISTANT:")
    print(answer)

    print("\n" + "=" * 60)

In [24]:
chat(
    message="My current project is Agentic RAG.",
    user_id="sunny_123",
    thread_id="thread_1"
)

NameError: name 'model' is not defined

In [25]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0
)

In [26]:
print(model)

metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}} profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000021C8A9EA8D0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021C8CAED820> root_client=<openai.OpenAI object at 0x0000021C8C764080> root_async_c

In [28]:
chat(
    message="My current project is Agentic RAG.",
    user_id="sunny_123",
    thread_id="thread_1"
)

OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: "sk-proj*****************************************************************************************************************************************************************paid. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
chat(
    message="What is my favorite programming language?",
    user_id="sunny_123",
    thread_id="thread_2"
)

In [ ]:
chat(
    message="Remember: my favorite language is Python.",
    user_id="sunny_123",
    thread_id="sunny_thread_1"
)

In [ ]:
chat(
    message="Remember: my favorite language is Java.",
    user_id="rahul_456",
    thread_id="rahul_thread_1"
)

In [ ]:
chat(
    message="What is my favorite programming language?",
    user_id="rahul_456",
    thread_id="rahul_thread_2"
)

In [ ]:
chat(
    message="What is my favorite programming language?",
    user_id="sunny_123",
    thread_id="sunny_thread_2"
)

In [ ]:

user_id = "sunny_123"

In [ ]:
namespace = (
    "users",
    user_id,
    "memories"
)

In [ ]:
memories = store.search(namespace)

for memory in memories:
    print(memory.value)

In [ ]:
memories = store.search(
    ("users", "sunny_123", "memories")
)

for memory in memories:
    print(memory.value["data"])